# Notebook 5 — Historical Change Detection

Compares vegetation and tree cover between two time periods and produces:
- NDVI difference map (pixel-level change)
- Change Vector Analysis (CVA) with gain/loss classification
- Post-Classification Comparison (PCC) confusion matrix
- Estimated tree count change
- Interactive Folium map with gain/loss overlay
- Summary chart and JSON report

**Methods**:
- NDVI differencing — Singh (1989)
- CVA — Malila (1980)
- PCC — Jensen (1996)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
import yaml
from pathlib import Path
from scipy.ndimage import gaussian_filter

from src.analysis.indices import ndvi, evi2
from src.analysis.change_detection import (
    ndvi_difference,
    change_vector_analysis,
    classify_vegetation,
    post_classification_comparison,
    vegetation_change_summary,
)
from src.visualization.maps import (
    create_base_map, ndvi_to_rgb, add_raster_overlay,
    add_change_map, save_map, create_change_summary_chart,
)

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

Path('../results').mkdir(exist_ok=True)
print('Configuration loaded.')

In [ ]:
# --- Simulate baseline and current NDVI for demonstration ---
# Replace with actual read_bands() calls on your downloaded GeoTIFFs

np.random.seed(7)
H, W = 300, 300
pixel_size_m = 10.0   # Sentinel-2 resolution
pixel_area_m2 = pixel_size_m ** 2

# Baseline: healthier, denser vegetation
ndvi_baseline = np.zeros((H, W), dtype=np.float32)
for _ in range(200):
    r = np.random.randint(5, H-5)
    c = np.random.randint(5, W-5)
    crown = np.zeros((H, W))
    crown[r, c] = np.random.uniform(0.55, 0.85)
    ndvi_baseline += gaussian_filter(crown, sigma=np.random.uniform(3, 7))
ndvi_baseline = np.clip(ndvi_baseline + np.random.normal(0.1, 0.05, (H, W)), 0, 1)

# Current: some deforestation (bottom-right quadrant), some regrowth (top-left)
ndvi_current = ndvi_baseline.copy()
# Simulate loss: reduce NDVI in a cleared zone
ndvi_current[180:260, 180:260] *= 0.35  # heavy loss
ndvi_current[200:240, 200:240] = np.random.uniform(0.05, 0.12, (40, 40))  # bare soil
# Simulate gain: plantation in top-left
ndvi_current[20:80, 20:80] = np.clip(
    ndvi_current[20:80, 20:80] + np.random.uniform(0.1, 0.25, (60, 60)), 0, 1
)
ndvi_current += np.random.normal(0, 0.02, (H, W))
ndvi_current = np.clip(ndvi_current, 0, 1)

print(f'Baseline NDVI — mean: {ndvi_baseline.mean():.3f}, std: {ndvi_baseline.std():.3f}')
print(f'Current NDVI  — mean: {ndvi_current.mean():.3f}, std: {ndvi_current.std():.3f}')

## Method 1 — NDVI Differencing

ΔV = NDVI_current - NDVI_baseline

Significant change defined as |ΔV| > 1.5 × std(ΔV).

**Reference**: Singh (1989). *International Journal of Remote Sensing*, 10(6).

In [ ]:
diff = ndvi_difference(ndvi_baseline, ndvi_current, threshold_std=1.5)

print('NDVI Differencing results:')
print(f'  Gain pixels: {diff["gain_mask"].sum():,} ({diff["gain_mask"].mean()*100:.1f}%)')
print(f'  Loss pixels: {diff["loss_mask"].sum():,} ({diff["loss_mask"].mean()*100:.1f}%)')
print(f'  Threshold:   ±{diff["threshold"]:.4f} NDVI units')
gain_ha = diff['gain_mask'].sum() * pixel_area_m2 / 10_000
loss_ha = diff['loss_mask'].sum() * pixel_area_m2 / 10_000
print(f'  Gain area:   {gain_ha:.2f} ha')
print(f'  Loss area:   {loss_ha:.2f} ha')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('NDVI Change Detection', fontsize=13, fontweight='bold')

axes[0].imshow(ndvi_baseline, cmap='RdYlGn', vmin=0, vmax=0.9)
axes[0].set_title(f'NDVI Baseline\n({cfg["dates"]["baseline"]["label"]})')

axes[1].imshow(ndvi_current, cmap='RdYlGn', vmin=0, vmax=0.9)
axes[1].set_title(f'NDVI Current\n({cfg["dates"]["current"]["label"]})')

im = axes[2].imshow(diff['delta'], cmap='RdBu', vmin=-0.4, vmax=0.4)
axes[2].set_title('ΔNDVI (red=loss, blue=gain)')
plt.colorbar(im, ax=axes[2])

change_rgb = np.zeros((H, W, 3), dtype=np.uint8)
change_rgb[diff['gain_mask']] = [0, 180, 0]
change_rgb[diff['loss_mask']] = [220, 0, 0]
axes[3].imshow(change_rgb)
axes[3].set_title(f'Change Map\nGain={gain_ha:.1f}ha, Loss={loss_ha:.1f}ha')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig('../results/ndvi_change_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## Method 2 — Change Vector Analysis (CVA)

Uses NDVI + EVI2 as a 2D feature space.
Classifies significant change as vegetation gain or loss based on
the direction of the change vector.

**Reference**: Malila, W.A. (1980). *LARS Symposia*, Paper 385.

In [ ]:
# Compute EVI2 (two-band EVI — no blue required)
# Using simulated NIR and Red arrays derived from NDVI for demo
# In practice, derive from actual band arrays
nir_sim = np.clip(ndvi_baseline * 0.4 + 0.3 + np.random.normal(0, 0.02, (H, W)), 0, 1)
red_sim = np.clip(nir_sim - ndvi_baseline * (nir_sim + nir_sim * 0.8), 0, 1)
evi2_baseline = evi2(nir_sim, red_sim)

nir_cur = np.clip(ndvi_current * 0.4 + 0.3 + np.random.normal(0, 0.02, (H, W)), 0, 1)
red_cur = np.clip(nir_cur - ndvi_current * (nir_cur + nir_cur * 0.8), 0, 1)
evi2_current = evi2(nir_cur, red_cur)

# Stack into feature arrays: (2, H, W)
features_t1 = np.stack([ndvi_baseline, evi2_baseline], axis=0)
features_t2 = np.stack([ndvi_current,  evi2_current],  axis=0)

cva = change_vector_analysis(features_t1, features_t2, magnitude_threshold_std=1.5)

print('CVA Results:')
type_names = {0: 'No change', 1: 'Veg gain', 2: 'Veg loss', 3: 'Other'}
for code, name in type_names.items():
    px = (cva['change_type'] == code).sum()
    ha = px * pixel_area_m2 / 10_000
    print(f'  {name:12s}: {px:6,} px ({ha:.2f} ha)')

## Method 3 — Post-Classification Comparison (PCC)

In [ ]:
classes_t1 = classify_vegetation(ndvi_baseline)
classes_t2 = classify_vegetation(ndvi_current)

pcc = post_classification_comparison(classes_t1, classes_t2, pixel_area_m2)

print('Post-Classification Comparison:')
print(pcc['summary_text'])

# Plot transition matrix
matrix = pcc['from_to_matrix']
class_names = list(pcc['class_names'].values())
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matrix, cmap='Blues')
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels([f'{n}\n(current)' for n in class_names], fontsize=9)
ax.set_yticklabels([f'{n}\n(baseline)' for n in class_names], fontsize=9)
ax.set_xlabel('Current class')
ax.set_ylabel('Baseline class')
ax.set_title('Land Cover Transition Matrix (pixels)')
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{matrix[i,j]:,}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('../results/transition_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Full Change Summary with Tree Count

In [ ]:
summary = vegetation_change_summary(
    ndvi_t1=ndvi_baseline,
    ndvi_t2=ndvi_current,
    pixel_area_m2=pixel_area_m2,
    ndvi_threshold=cfg['analysis']['vegetation_threshold'],
    mean_crown_area_m2=cfg['analysis']['mean_crown_area_m2'],
)

print('=' * 55)
print('  VEGETATION & TREE CHANGE SUMMARY')
print('=' * 55)
print(f'  Study area:          {summary["total_area_ha"]:,.1f} ha')
print(f'  Vegetation — past:   {summary["veg_area_t1_ha"]:,.2f} ha')
print(f'  Vegetation — now:    {summary["veg_area_t2_ha"]:,.2f} ha')
print(f'  Change:              {summary["veg_change_ha"]:+,.2f} ha ({summary["veg_change_pct"]:+.1f}%)')
print(f'  Estimated trees (past): {summary["estimated_trees_t1"]:,}')
print(f'  Estimated trees (now):  {summary["estimated_trees_t2"]:,}')
print(f'  Tree count delta:       {summary["estimated_trees_delta"]:+,}')
print(f'  Mean NDVI past: {summary["mean_ndvi_t1"]:.4f}')
print(f'  Mean NDVI now:  {summary["mean_ndvi_t2"]:.4f}')
print('=' * 55)

In [ ]:
# Generate summary chart
create_change_summary_chart(summary, output_path='../results/change_summary.png')

In [ ]:
# Save full report
bbox = cfg['aoi']['bbox']
report = {
    'aoi': cfg['aoi']['name'],
    'bbox': bbox,
    'baseline': cfg['dates']['baseline']['label'],
    'current': cfg['dates']['current']['label'],
    'source': cfg['source']['primary'],
    'pixel_size_m': pixel_size_m,
    'summary': summary,
    'cva': {
        'gain_ha': float((cva['change_type'] == 1).sum() * pixel_area_m2 / 10_000),
        'loss_ha': float((cva['change_type'] == 2).sum() * pixel_area_m2 / 10_000),
        'threshold_magnitude': float(cva['threshold']),
    },
    'pcc': {
        'forest_gain_ha': pcc['forest_gain_ha'],
        'forest_loss_ha': pcc['forest_loss_ha'],
        'net_change_ha': pcc['net_change_ha'],
    },
}
with open('../results/full_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('Full report saved to results/full_report.json')

In [ ]:
# Generate interactive Folium map
# (Replace bbox with real coordinates for a georeferenced map)
center = ((bbox[1] + bbox[3]) / 2, (bbox[0] + bbox[2]) / 2)
m = create_base_map(center, zoom=13)

map_bounds = [[bbox[1], bbox[0]], [bbox[3], bbox[2]]]
add_raster_overlay(m, ndvi_to_rgb(ndvi_baseline), map_bounds,
                   name=f'NDVI {cfg["dates"]["baseline"]["label"]}', opacity=0.7)
add_raster_overlay(m, ndvi_to_rgb(ndvi_current), map_bounds,
                   name=f'NDVI {cfg["dates"]["current"]["label"]}', opacity=0.7)
add_change_map(m, diff['gain_mask'], diff['loss_mask'], map_bounds)

map_path = save_map(m, '../results/change_map.html')
print(f'Open {map_path} in a browser to explore the interactive map.')